## UAV Intrusion Detection Stacked Ensemble
This notebook implements a meta-classifier ensemble using One-Class SVM, Autoencoder, and Local Outlier Factor as described in the Whelan et al. (2020) paper into the UAV Attack Dataset by Jason Whelan, et al.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import os
import gc

In [2]:
# Constants
RANDOM_STATE = 1082

base_dir = "../../../Datasets/csv/UAVATTACK/"
real_data_path = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"

models_list = ["ctgan", "gc", "tvae"]
sample_sizes = [10_000, 100_000, 1_000_000]

In [3]:
synthetic_dfs = {}
# Load synthetic datasets for each model/sample size
for model in models_list:
    for n in sample_sizes:
        file_name = f"spoofing_{model}_{n}.csv"
        file_path = os.path.join(base_dir, file_name)
        try:
            synthetic_dfs[file_name] = pd.read_csv(file_path)
            print(f"Loaded {file_name}")
        except Exception as e:
            print(f"Could not load {file_name}: {e}")

# Load real data
try:
    real_df = pd.read_csv(real_data_path)
    real_df.columns = real_df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
    print("Loaded real data")
except Exception as e:
    print(f"Could not load real data: {e}")
    real_df = None


Loaded spoofing_ctgan_10000.csv
Loaded spoofing_ctgan_100000.csv
Loaded spoofing_ctgan_1000000.csv
Loaded spoofing_gc_10000.csv
Loaded spoofing_gc_100000.csv
Loaded spoofing_gc_1000000.csv
Loaded spoofing_tvae_10000.csv
Loaded spoofing_tvae_100000.csv
Loaded spoofing_tvae_1000000.csv
Loaded real data


In [4]:
if 'label' not in real_df.columns:
    raise ValueError("Real data does not have 'label' column.")
X_real = real_df.drop(columns=[col for col in ['label', 'timestamp'] if col in real_df.columns])
y_real = real_df['label']
if y_real.dtype == object or y_real.dtype.name == 'category':
    le_real = LabelEncoder()
    y_real = le_real.fit_transform(y_real)
X_real = X_real.astype(np.float32)
del real_df
gc.collect()

72

In [5]:
# Stacked ensemble for each synthetic dataset
for file_name, df in synthetic_dfs.items():
    if 'label' not in df.columns:
        print(f"Warning: 'label' column not found in {file_name}. Skipping.")
        continue

    # Clean synthetic columns to match real data's format
    df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
    # Drop timestamp if present
    X_synth_full = df.drop(columns=[col for col in ['label', 'timestamp'] if col in df.columns])
    y_synth_full = df['label']

    # Align columns between synthetic and real data (order and names)
    common_cols = [col for col in X_real.columns if col in X_synth_full.columns]
    X_synth_full = X_synth_full[common_cols]
    X_real_aligned = X_real[common_cols]

    # Use only benign samples from synthetic for training base detectors
    X_train_raw = X_synth_full[y_synth_full == 0]
    if len(X_train_raw) < 10:
        print(f"Too few benign samples in {file_name}, skipping.")
        continue

    # Normalize using only synthetic benign training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_synth_full_scaled = scaler.transform(X_synth_full)
    X_real_scaled = scaler.transform(X_real_aligned)

    # Clean up raw data to free memory
    del X_train_raw, X_synth_full, X_real_aligned
    gc.collect()

    # PCA for OC-SVM and LOF (85% variance)
    pca_svm_lof = PCA(n_components=0.85, random_state=RANDOM_STATE)
    X_train_pca_svm_lof = pca_svm_lof.fit_transform(X_train_scaled)
    X_synth_full_pca_svm_lof = pca_svm_lof.transform(X_synth_full_scaled)
    X_real_pca_svm_lof = pca_svm_lof.transform(X_real_scaled)

    # PCA for Autoencoder (95% variance)
    pca_ae = PCA(n_components=0.95, random_state=RANDOM_STATE)
    X_train_pca_ae = pca_ae.fit_transform(X_train_scaled)
    X_synth_full_pca_ae = pca_ae.transform(X_synth_full_scaled)
    X_real_pca_ae = pca_ae.transform(X_real_scaled)

    # Free up scaled data
    del X_train_scaled, X_synth_full_scaled, X_real_scaled
    gc.collect()

    # One-Class SVM
    ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
    ocsvm.fit(X_train_pca_svm_lof)
    ocsvm_score_synth = ocsvm.decision_function(X_synth_full_pca_svm_lof)
    ocsvm_score_real = ocsvm.decision_function(X_real_pca_svm_lof)

    # Local Outlier Factor
    lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
    lof.fit(X_train_pca_svm_lof)
    lof_score_synth = lof.decision_function(X_synth_full_pca_svm_lof)
    lof_score_real = lof.decision_function(X_real_pca_svm_lof)

    # Free up PCA'd for SVM/LOF
    del X_train_pca_svm_lof, X_synth_full_pca_svm_lof, X_real_pca_svm_lof
    gc.collect()

    # Autoencoder using MLP
    ae = MLPRegressor(hidden_layer_sizes=(100, 50), activation='tanh', max_iter=1000, random_state=RANDOM_STATE, learning_rate='adaptive',early_stopping=True,)
    ae.fit(X_train_pca_ae, X_train_pca_ae)
    recon_synth = ae.predict(X_synth_full_pca_ae)
    recon_real = ae.predict(X_real_pca_ae)
    ae_score_synth = np.mean((X_synth_full_pca_ae - recon_synth) ** 2, axis=1)
    ae_score_real = np.mean((X_real_pca_ae - recon_real) ** 2, axis=1)

    # Free up PCA'd for Autoencoder and reconstructions
    del X_train_pca_ae, X_synth_full_pca_ae, X_real_pca_ae, recon_synth, recon_real
    gc.collect()

    # Stack anomaly scores as features
    stacked_features_synth = np.vstack([ocsvm_score_synth, lof_score_synth, ae_score_synth]).T
    stacked_features_real = np.vstack([ocsvm_score_real, lof_score_real, ae_score_real]).T

    # Meta-classifier (Random Forest) trained on synthetic, evaluated on real
    meta_clf = RandomForestClassifier(random_state=RANDOM_STATE)
    meta_clf.fit(stacked_features_synth, y_synth_full)
    stacked_pred = meta_clf.predict(stacked_features_real)

    # Clean up features and y
    del stacked_features_synth, stacked_features_real, y_synth_full
    gc.collect()

    print(f"\nTraining on synthetic: {file_name}, testing on real data")
    print(classification_report(
        y_real,
        stacked_pred,
        labels=[0, 1],
        target_names=["benign", "malicious"],
        zero_division=0
    ))


Training on synthetic: spoofing_ctgan_10000.csv, testing on real data
              precision    recall  f1-score   support

      benign       0.99      0.85      0.92      3124
   malicious       0.51      0.97      0.67       498

    accuracy                           0.87      3622
   macro avg       0.75      0.91      0.80      3622
weighted avg       0.93      0.87      0.89      3622


Training on synthetic: spoofing_ctgan_100000.csv, testing on real data
              precision    recall  f1-score   support

      benign       0.97      0.93      0.95      3124
   malicious       0.65      0.83      0.73       498

    accuracy                           0.92      3622
   macro avg       0.81      0.88      0.84      3622
weighted avg       0.93      0.92      0.92      3622


Training on synthetic: spoofing_ctgan_1000000.csv, testing on real data
              precision    recall  f1-score   support

      benign       0.96      0.95      0.96      3124
   malicious       0.